# PopOut -- Pesquisa Adversarial com MCTS e Árvores de Decisão
**Inteligência Artificial 2025/2026** · Grupo 3

---

## 1. Introdução

Este notebook documenta o projeto desenvolvido no âmbito da unidade curricular de Inteligência Artificial. O trabalho divide-se em duas grandes componentes: a implementação de um agente capaz de jogar **PopOut** utilizando o algoritmo **Monte Carlo Tree Search (MCTS)**, e a construção de **árvores de decisão** treinadas pelo algoritmo **ID3** sobre datasets gerados a partir do próprio jogo.

### 1.1 O que é o PopOut?

O PopOut é uma variante do clássico Connect-4. Para além da jogada habitual de *drop* (largar um disco no topo de uma coluna), cada jogador pode também realizar um *pop* — remover um disco **seu** da **linha inferior** de qualquer coluna. Quando isto acontece, todos os discos acima descem uma posição, alterando potencialmente toda a configuração do tabuleiro.

| Regra | Descrição |
|-------|-----------|
| Drop | Colocar um disco no topo de uma coluna (a gravidade fá-lo cair até à célula vazia mais baixa) |
| Pop | Remover um disco **próprio** da linha inferior; todos os discos acima descem uma posição |
| Vitória | O primeiro jogador a alinhar 4 discos (horizontal / vertical / diagonal) vence |
| Regra especial 1 | Se um *pop* cria 4-em-linha para **ambos** os jogadores, quem fez o *pop* vence |
| Regra especial 2 | Se o tabuleiro está **cheio** e não é possível *drop*, o jogador pode fazer *pop* (se possível) ou declarar **empate** |
| Regra especial 3 | Se o mesmo estado do tabuleiro se repetir **3 vezes**, qualquer jogador pode declarar **empate** |

A introdução do *pop* torna o PopOut consideravelmente mais complexo do que o Connect-4 tradicional: o número de jogadas legais por turno é maior, e a possibilidade de remover peças faz com que posições aparentemente seguras se tornem vulneráveis. Esta complexidade adicional torna-o um problema particularmente interessante para algoritmos de pesquisa adversarial.

### 1.2 Estrutura do Projeto

| Ficheiro | Conteúdo |
|----------|-----------|
| `popout_game.py` | Classe `PopOutGame` — toda a lógica do jogo, aplicação de jogadas, deteção de vitória |
| `mcts.py` | `MCTSNode`, `MCTS` e funções auxiliares para as diferentes variantes |
| `main.py` | Runner interativo por texto (Humano vs Humano / Humano vs IA / IA vs IA) |
| `Relatório.ipynb` | **Este notebook** — documentação, experiências, geração de datasets, árvores de decisão |

---
## 2. Implementação do Jogo PopOut

A classe `PopOutGame` encapsula toda a lógica do jogo. Optámos por um design **imutável**: cada jogada devolve um **novo** estado, sem alterar o original. Esta decisão facilita a integração com o MCTS, onde se criam e exploram múltiplas ramificações sem risco de corromper o estado de referência.

### 2.1 Decisões de Design

- **Representação do tabuleiro:** Matriz `6×7` de inteiros (`0` = vazio, `1` = Jogador 1, `2` = Jogador 2). A linha 0 corresponde ao topo.
- **Histórico de estados:** Um dicionário que mapeia cada configuração `(tabuleiro, jogador_atual)` ao número de ocorrências. Permite verificar a regra de empate por repetição sem percorrer todo o histórico de jogadas.
- **Cópia eficiente:** O método `_copy()` usa `object.__new__` para evitar a re-execução do `__init__`, copiando apenas os atributos necessários. Isto é relevante porque o MCTS cria milhares de cópias por decisão.

### 2.2 Funções Principais

- **`get_all_moves()`** — Devolve todas as jogadas legais, combinando *drops* (colunas com espaço) e *pops* (colunas onde o jogador atual tem disco na base).
- **`apply_move(move_type, col)`** — Cria uma cópia do estado, aplica a jogada, e resolve o resultado (vitória, empate, ou troca de turno).
- **`_check_four_in_row(player)`** — Verifica as quatro direções possíveis (horizontal, vertical, e duas diagonais) para detetar alinhamentos de 4 peças.
- **`_resolve_after_pop()`** — Implementa a regra especial: se ambos os jogadores ficam com 4-em-linha após um *pop*, quem fez o *pop* vence.

In [1]:
# Toda a lógica do jogo está em popout_game.py
# Esta célula redefine a classe inline para que o notebook seja auto-contido.

import math, random, time, csv, os
from copy import deepcopy

EMPTY   = 0
PLAYER1 = 1   # 'X'
PLAYER2 = 2   # 'O'

class PopOutGame:
    """
    Estado de jogo imutável para PopOut.
    Todas as jogadas devolvem um NOVO estado; o original nunca é modificado.
    """
    EMPTY = EMPTY; PLAYER1 = PLAYER1; PLAYER2 = PLAYER2

    def __init__(self, rows=6, cols=7):
        self.rows = rows
        self.cols = cols
        self.board = [[EMPTY]*cols for _ in range(rows)]
        self.current_player = PLAYER1
        self.state_history = {}
        self.winner = None
        self.game_over = False
        self._record_state()

    # ── Rastreio de estados ──────────────────────────────────────────────────
    def _get_state_key(self):
        return (tuple(tuple(r) for r in self.board), self.current_player)
    def _record_state(self):
        k = self._get_state_key()
        self.state_history[k] = self.state_history.get(k, 0) + 1
    def get_state_repetitions(self):
        return self.state_history.get(self._get_state_key(), 0)
    def is_repetition_draw_available(self):
        return self.get_state_repetitions() >= 3

    # ── Consultas ao tabuleiro ────────────────────────────────────────────────
    def is_board_full(self):
        return all(self.board[0][c] != EMPTY for c in range(self.cols))
    def get_drop_moves(self):
        return [c for c in range(self.cols) if self.board[0][c] == EMPTY]
    def get_pop_moves(self):
        return [c for c in range(self.cols) if self.board[self.rows-1][c] == self.current_player]
    def get_all_moves(self):
        if self.game_over: return []
        return [('drop',c) for c in self.get_drop_moves()] + [('pop',c) for c in self.get_pop_moves()]

    # ── Aplicação de jogadas ──────────────────────────────────────────────────
    def apply_move(self, move_type, col):
        if self.game_over: return None
        g = self._copy()
        if move_type == 'drop':
            if g.board[0][col] != EMPTY: return None
            row = self.rows-1
            while row >= 0 and g.board[row][col] != EMPTY: row -= 1
            if row < 0: return None
            g.board[row][col] = self.current_player
            g._resolve_after_drop()
        elif move_type == 'pop':
            if g.board[self.rows-1][col] != self.current_player: return None
            for r in range(self.rows-1, 0, -1):
                g.board[r][col] = g.board[r-1][col]
            g.board[0][col] = EMPTY
            g._resolve_after_pop()
        else: return None
        return g

    def _resolve_after_drop(self):
        p1, p2 = self._check_four(PLAYER1), self._check_four(PLAYER2)
        if p1 or p2:
            self.winner = self.current_player if (p1 and p2) else (PLAYER1 if p1 else PLAYER2)
            self.game_over = True
        else:
            self._switch()

    def _resolve_after_pop(self):
        p1, p2 = self._check_four(PLAYER1), self._check_four(PLAYER2)
        if p1 or p2:
            self.winner = self.current_player  # quem fez pop vence em caso simultâneo
            self.game_over = True
        else:
            self._switch()

    def _switch(self):
        self.current_player = PLAYER2 if self.current_player == PLAYER1 else PLAYER1
        self._record_state()

    def _check_four(self, p):
        b, R, C = self.board, self.rows, self.cols
        for r in range(R):
            for c in range(C-3):
                if b[r][c]==b[r][c+1]==b[r][c+2]==b[r][c+3]==p: return True
        for r in range(R-3):
            for c in range(C):
                if b[r][c]==b[r+1][c]==b[r+2][c]==b[r+3][c]==p: return True
        for r in range(R-3):
            for c in range(C-3):
                if b[r][c]==b[r+1][c+1]==b[r+2][c+2]==b[r+3][c+3]==p: return True
        for r in range(R-3):
            for c in range(3,C):
                if b[r][c]==b[r+1][c-1]==b[r+2][c-2]==b[r+3][c-3]==p: return True
        return False

    def _copy(self):
        g = object.__new__(PopOutGame)
        g.rows=self.rows; g.cols=self.cols
        g.board=[r[:] for r in self.board]
        g.current_player=self.current_player
        g.state_history=dict(self.state_history)
        g.winner=self.winner; g.game_over=self.game_over
        return g

    # ── Visualização ─────────────────────────────────────────────────────────
    @staticmethod
    def player_symbol(p): return 'X' if p==PLAYER1 else 'O'
    def display(self):
        s = {EMPTY:'-', PLAYER1:'X', PLAYER2:'O'}
        print()
        for row in self.board: print(''.join(s[c] for c in row))
        print('1234567'[:self.cols])
        if not self.game_over:
            print(f"\nÉ a vez de {self.player_symbol(self.current_player)}.")
        elif self.winner: print(f"\n{self.player_symbol(self.winner)} vence!")
        else: print("\nEmpate!")
        print()
    def get_board_flat(self): return [c for row in self.board for c in row]

print('PopOutGame definido.')

PopOutGame definido.


### 2.3 Demonstração — algumas jogadas manuais

Para validar a implementação, executamos uma sequência de jogadas e verificamos que o tabuleiro, a troca de turnos e a deteção de jogadas legais funcionam corretamente.

In [2]:
g = PopOutGame()
# Largar alguns discos
for col in [3, 3, 3, 4, 2, 5]:
    g = g.apply_move('drop', col)
g.display()
print('Jogadas disponíveis:', g.get_all_moves())


-------
-------
-------
---X---
---O---
--XXOO-
1234567

É a vez de X.

Jogadas disponíveis: [('drop', 0), ('drop', 1), ('drop', 2), ('drop', 3), ('drop', 4), ('drop', 5), ('drop', 6), ('pop', 2), ('pop', 3)]


In [3]:
# Demonstração da jogada pop: preencher a coluna 3 e depois fazer pop
g2 = PopOutGame()
for col in [3,3,3,3,3,3]:
    g2 = g2.apply_move('drop', col)
    if g2 is None:
        print('Coluna cheia'); break
if g2:
    g2.display()
    print('Jogadas pop disponíveis para X:', g2.get_pop_moves())
    g3 = g2.apply_move('pop', 3)
    if g3:
        print('Após pop na coluna 3:')
        g3.display()


---O---
---X---
---O---
---X---
---O---
---X---
1234567

É a vez de X.

Jogadas pop disponíveis para X: [3]
Após pop na coluna 3:

-------
---O---
---X---
---O---
---X---
---O---
1234567

É a vez de O.



---
## 3. Monte Carlo Tree Search (MCTS)

O MCTS é um algoritmo de pesquisa adversarial que utiliza **simulação aleatória (Monte Carlo)** para estimar o valor de cada jogada sem necessitar de uma função de avaliação manual. Este é o principal algoritmo utilizado neste projeto para dotar a IA de capacidade de jogo.

### 3.1 As Quatro Fases

O algoritmo opera em ciclos de quatro fases:

1. **Seleção** — A partir da raiz, percorre-se a árvore escolhendo sucessivamente o filho que maximiza o valor **UCT**, até se atingir um nó que não esteja totalmente expandido (ou um nó terminal).

2. **Expansão** — Adiciona-se um novo nó filho correspondente a uma jogada ainda não explorada. É aqui que a árvore cresce.

3. **Simulação (Rollout)** — A partir do novo nó, joga-se aleatoriamente (ou com heurística) até ao final do jogo. O resultado deste "jogo rápido" serve de estimativa da qualidade da posição.

4. **Retropropagação** — O resultado da simulação é propagado de volta até à raiz, atualizando contagens de visitas e somas de vitórias ao longo de todo o caminho.

### 3.2 Fórmula UCT

A seleção do filho a explorar é governada pelo **Upper Confidence Bound for Trees (UCT)**:

$$\text{UCT}(v') = \underbrace{\frac{Q(v')}{N(v')}}_{\text{exploração dos ganhos}} + C \cdot \underbrace{\sqrt{\frac{\ln N(v)}{N(v')}}}_{\text{exploração do desconhecido}}$$

onde:
- $Q(v')$ = vitórias acumuladas no filho $v'$ (da perspetiva do **jogador do filho**)
- $N(v')$ = número de visitas ao filho $v'$
- $N(v)$ = número de visitas ao pai $v$
- $C$ = constante de exploração (tipicamente $\sqrt{2} \approx 1.414$)

Na nossa implementação, como $Q/N$ está armazenado da perspetiva do jogador do **filho** (que é o adversário do jogador do nó atual), o pai seleciona usando $(1 - Q/N) + C\sqrt{\ldots}$, invertendo a perspetiva para a do jogador corrente.

O parâmetro $C$ controla o equilíbrio entre **exploitation** (explorar jogadas já conhecidas como boas) e **exploration** (visitar jogadas menos testadas). Um $C$ alto diversifica a pesquisa; um $C$ baixo concentra-a nas melhores opções conhecidas.

### 3.3 Variantes Implementadas

Para além da implementação base, explorámos seis configurações distintas do MCTS para compreender o impacto de diferentes estratégias:

| Variante | Diferença principal |
|----------|---------------------|
| Standard UCT | $C = \sqrt{2}$ — configuração clássica da literatura |
| Alta exploração | $C = 2.5$ — árvore mais larga, pesquisa menos profunda |
| Baixa exploração (greedy) | $C = 0.5$ — foco quase exclusivo em jogadas promissoras |
| Progressive widening | Limite de filhos por nó (`max_children`) — reduz o branching factor |
| Heuristic rollout | Look-ahead de 1 passo durante simulação: prefere vitórias e bloqueios imediatos |
| UCT-Tuned | Substitui o termo de exploração por um *bound* baseado na variância empírica |

### 3.4 Implementação

A implementação está dividida em duas classes: `MCTSNode` (cada nó da árvore de pesquisa) e `MCTS` (o algoritmo propriamente dito).

**`MCTSNode`** armazena:
- O estado do jogo nesse ponto
- Referência ao pai e à jogada que levou a este nó
- Lista de filhos já expandidos
- Estatísticas: vitórias acumuladas (`wins`) e visitas (`visits`)
- Jogadas ainda não experimentadas (`untried_moves`)

Utilizamos `__slots__` para reduzir o consumo de memória — relevante quando a árvore cresce para dezenas de milhares de nós.

**`MCTS`** recebe como parâmetros o número de iterações, a constante $C$, o limite de filhos (progressive widening), a estratégia de rollout e a flag para UCT-Tuned. O método principal `get_best_move` executa o ciclo completo e devolve a jogada com mais visitas na raiz (política *robust child*).

In [4]:
# ─── Nó MCTS ─────────────────────────────────────────────────────────────────

class MCTSNode:
    __slots__ = ('game_state','parent','move','children','wins','visits','untried_moves')

    def __init__(self, game_state, parent=None, move=None):
        self.game_state    = game_state
        self.parent        = parent
        self.move          = move
        self.children      = []
        self.wins          = 0.0
        self.visits        = 0
        self.untried_moves = list(game_state.get_all_moves())

    def is_terminal(self):       return self.game_state.game_over
    def is_fully_expanded(self): return len(self.untried_moves) == 0

    def uct_value(self, C):
        """UCT da perspetiva do pai: (1 - Q/N) + C*sqrt(ln(parent_N)/N)"""
        if self.visits == 0: return float('inf')
        return (1.0 - self.wins/self.visits) + C * math.sqrt(math.log(self.parent.visits)/self.visits)

    def uct_tuned_value(self, C):
        """UCT-Tuned com bound de variância empírica."""
        if self.visits == 0: return float('inf')
        q = self.wins / self.visits
        var = q - q*q + math.sqrt(2*math.log(self.parent.visits)/self.visits)
        return (1.0-q) + C*math.sqrt(math.log(self.parent.visits)/self.visits * min(0.25,var))

    def best_child(self, C, tuned=False):
        if tuned:
            return max(self.children, key=lambda ch: ch.uct_tuned_value(C))
        return max(self.children, key=lambda ch: ch.uct_value(C))

# ─── MCTS ────────────────────────────────────────────────────────────────────

class MCTS:
    def __init__(self, iterations=1000, C=math.sqrt(2),
                 max_children=None, rollout='random', tuned=False, name='MCTS'):
        self.iterations   = iterations
        self.C            = C
        self.max_children = max_children
        self.rollout      = rollout
        self.tuned        = tuned
        self.name         = name

    def get_best_move(self, state):
        root = MCTSNode(state)
        for _ in range(self.iterations):
            leaf = self._select(root)
            if not leaf.is_terminal():
                leaf = self._expand(leaf)
            result = self._simulate(leaf)
            self._backpropagate(leaf, result)
        if not root.children:
            moves = state.get_all_moves()
            return random.choice(moves) if moves else None
        return max(root.children, key=lambda ch: ch.visits).move

    def get_move_stats(self, state):
        root = MCTSNode(state)
        for _ in range(self.iterations):
            leaf = self._select(root)
            if not leaf.is_terminal(): leaf = self._expand(leaf)
            self._backpropagate(leaf, self._simulate(leaf))
        return {ch.move: {'visits':ch.visits,
                          'win_rate': ch.wins/ch.visits if ch.visits else 0.0}
                for ch in root.children}

    def _select(self, node):
        while not node.is_terminal() and node.is_fully_expanded():
            node = node.best_child(self.C, self.tuned)
        return node

    def _expand(self, node):
        if not node.untried_moves: return node
        if self.max_children and len(node.children) >= self.max_children:
            return node.best_child(self.C, self.tuned) if node.children else node
        move = random.choice(node.untried_moves)
        ns   = node.game_state.apply_move(*move)
        if ns is None:
            node.untried_moves.remove(move)
            return self._expand(node)
        child = MCTSNode(ns, parent=node, move=move)
        node.untried_moves.remove(move)
        node.children.append(child)
        return child

    def _simulate(self, node):
        state  = node.game_state._copy()
        player = node.game_state.current_player
        for _ in range(200):
            if state.game_over: break
            moves = state.get_all_moves()
            if not moves: break
            move = self._heuristic_select(state, moves) if self.rollout=='heuristic' else random.choice(moves)
            ns   = state.apply_move(*move)
            if ns: state = ns
        if state.winner == player: return 1.0
        if state.winner is None:   return 0.5
        return 0.0

    def _heuristic_select(self, state, moves):
        p = state.current_player
        opp = PLAYER2 if p==PLAYER1 else PLAYER1
        for m in moves:
            s = state.apply_move(*m)
            if s and s.winner == p: return m
        for m in moves:
            s = state.apply_move(*m)
            if s and s.winner == opp: return m
        return random.choice(moves)

    def _backpropagate(self, node, result):
        while node:
            node.visits += 1
            node.wins   += result
            result = 1.0 - result
            node = node.parent

print('MCTS definido.')

MCTS definido.


### 3.5 Teste de sanidade — a IA escolhe a jogada vencedora

Antes de avançar para experiências mais complexas, é fundamental verificar que o MCTS funciona corretamente num cenário simples: quando existe uma vitória imediata disponível, o algoritmo deve encontrá-la consistentemente, mesmo com poucas iterações.

In [5]:
# Configurar um estado em que X (PLAYER1) pode vencer imediatamente
g = PopOutGame()
for c in [0,1,1,2,2,3]:   # X tem 3 em linha na base
    g = g.apply_move('drop', c)
g.display()

ai = MCTS(iterations=500, C=math.sqrt(2), name='Teste-Sanidade')
move = ai.get_best_move(g)
print(f'IA escolheu: {move[0].upper()} coluna {move[1]+1}')
g2 = g.apply_move(*move)
g2.display()


-------
-------
-------
-------
-XX----
XOOO---
1234567

É a vez de X.

IA escolheu: DROP coluna 5

-------
-------
-------
-------
-XX----
XOOOX--
1234567

É a vez de O.



---
## 4. Modos de Jogo

O enunciado exige três modos de jogo: **humano vs humano**, **humano vs computador**, e **computador vs computador**. A implementação interativa encontra-se em `main.py`, onde o jogador pode escolher o modo e o algoritmo. Para jogos automatizados (IA vs IA), usamos o helper `auto_play` que executa um jogo completo entre dois agentes MCTS.

O modo IA vs IA é particularmente útil para:
- Comparar variantes do MCTS de forma controlada
- Gerar datasets para as árvores de decisão
- Recolher estatísticas sobre desempenho relativo

In [6]:
def auto_play(ai1, ai2, verbose=False):
    """
    Executa um jogo completo entre dois agentes MCTS.
    Devolve (vencedor, n_jogadas). vencedor é PLAYER1, PLAYER2, ou None (empate).
    """
    game = PopOutGame()
    n = 0
    while not game.game_over:
        if game.is_repetition_draw_available():
            if verbose: print('[Empate por repetição]')
            return None, n
        ai = ai1 if game.current_player == PLAYER1 else ai2
        move = ai.get_best_move(game)
        if move is None: break
        ng = game.apply_move(*move)
        if ng is None: break
        game = ng
        n += 1
    if verbose: game.display()
    return game.winner, n

# Demonstração rápida: um jogo IA vs IA
ai_x = MCTS(iterations=300, C=math.sqrt(2), name='X-Standard')
ai_o = MCTS(iterations=300, C=math.sqrt(2), name='O-Standard')

winner, moves = auto_play(ai_x, ai_o, verbose=True)
sym = PopOutGame.player_symbol(winner) if winner else 'Empate'
print(f'Resultado: {sym}  |  Jogadas: {moves}')


OOOOOOX
OXOXOOX
XXOXXXO
OXXXOOO
XOOOXOX
OXOOXOO
1234567

O vence!

Resultado: O  |  Jogadas: 142


---
## 5. Experiências — Comparação de Variantes MCTS

Para compreender o impacto de cada variante, organizámos um **torneio round-robin**: cada par de variantes joga `N_GAMES` partidas, alternando quem começa como X. Esta alternância é importante porque no PopOut (tal como no Connect-4), o primeiro jogador tem uma ligeira vantagem estrutural.

As seis configurações testadas representam diferentes filosofias de pesquisa:
- **Standard** — a linha de base, com $C = \sqrt{2}$ conforme a literatura original
- **Alta exploração** — testa se vale a pena sacrificar profundidade por diversidade
- **Baixa exploração** — o oposto: foco máximo nas melhores jogadas conhecidas
- **Progressive widening** — limitar o número de filhos por nó pode ajudar quando o branching factor é elevado
- **Heuristic rollout** — substitui jogadas completamente aleatórias por um look-ahead de 1 passo
- **UCT-Tuned** — adapta a exploração com base na variância observada dos resultados

In [7]:
# Definir as variantes
ITERS = 400   # valor moderado para velocidade no notebook; aumentar para melhores estatísticas

variants = [
    MCTS(iterations=ITERS, C=math.sqrt(2),  name='Standard (C=sqrt2)'),
    MCTS(iterations=ITERS, C=2.5,           name='Alta-Expl (C=2.5)'),
    MCTS(iterations=ITERS, C=0.5,           name='Baixa-Expl (C=0.5)'),
    MCTS(iterations=ITERS, C=math.sqrt(2),  max_children=4, name='Prog-Widening(k=4)'),
    MCTS(iterations=ITERS, C=math.sqrt(2),  rollout='heuristic', name='Heuristic-Rollout'),
    MCTS(iterations=ITERS, C=math.sqrt(2),  tuned=True, name='UCT-Tuned'),
]
print('Variantes definidas:')
for v in variants: print(f'  {v.name}')

Variantes definidas:
  Standard (C=sqrt2)
  Alta-Expl (C=2.5)
  Baixa-Expl (C=0.5)
  Prog-Widening(k=4)
  Heuristic-Rollout
  UCT-Tuned


In [8]:
def tournament(variants, n_games=10):
    """
    Torneio round-robin: cada par joga n_games partidas (metade como X, metade como O).
    Devolve uma matriz W[i][j] = vitórias da variante i contra a variante j.
    """
    n = len(variants)
    W = [[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i == j: continue
            for game_idx in range(n_games):
                if game_idx % 2 == 0:
                    ai1, ai2 = variants[i], variants[j]
                    p_i = PLAYER1
                else:
                    ai1, ai2 = variants[j], variants[i]
                    p_i = PLAYER2
                winner, _ = auto_play(ai1, ai2)
                if winner == p_i:
                    W[i][j] += 1
    return W

N_GAMES = 6   # aumentar para resultados mais fiáveis
print(f'A executar torneio ({len(variants)} variantes x {N_GAMES} jogos por par)...')
t0 = time.time()
W = tournament(variants, n_games=N_GAMES)
print(f'Concluído em {time.time()-t0:.1f}s')

A executar torneio (6 variantes x 6 jogos por par)...
Concluído em 1030.3s


In [9]:
# Tabela de resultados
n = len(variants)
names = [v.name for v in variants]
totals = [sum(W[i]) for i in range(n)]

col_w = max(len(nm) for nm in names) + 2
header = f"{'Variante':<{col_w}}" + ''.join(f'{i:>4}' for i in range(n)) + '  TOTAL'
print(header)
print('-' * len(header))
for i, nm in enumerate(names):
    row = f'{nm:<{col_w}}' + ''.join(f'{W[i][j]:>4}' if j!=i else '   -' for j in range(n))
    row += f'  {totals[i]:>4}'
    print(row)
print()
best_idx = totals.index(max(totals))
print(f'Melhor variante: {names[best_idx]}  ({totals[best_idx]} vitórias)')

Variante               0   1   2   3   4   5  TOTAL
---------------------------------------------------
Standard (C=sqrt2)     -   0   3   0   4   0     7
Alta-Expl (C=2.5)      5   -   1   0   2   2    10
Baixa-Expl (C=0.5)     5   4   -   1   6   6    22
Prog-Widening(k=4)     6   6   6   -   5   6    29
Heuristic-Rollout      3   1   2   0   -   3     9
UCT-Tuned              4   6   2   0   4   -    16

Melhor variante: Prog-Widening(k=4)  (29 vitórias)


### 5.1 Efeito do orçamento de iterações

Um dos parâmetros mais óbvios do MCTS é o número de iterações — quantas vezes o ciclo seleção-expansão-simulação-retropropagação é executado. Naturalmente, mais iterações permitem explorar mais a árvore, mas a relação não é linear: há rendimentos decrescentes, especialmente em posições com poucos movimentos legais (final do jogo).

Testamos o MCTS Standard com diferentes orçamentos contra uma baseline fraca (50 iterações) para quantificar esta relação.

In [10]:
# Comparar MCTS Standard com diferentes orçamentos vs baseline fraca (50 iterações)
baseline = MCTS(iterations=50, C=math.sqrt(2), name='Fraco(50)')
budgets  = [100, 200, 400, 800]
REPS     = 8

print(f'Taxa de vitória do MCTS Standard(n) vs Fraco(50) em {REPS} jogos cada:')
for n_iter in budgets:
    strong = MCTS(iterations=n_iter, C=math.sqrt(2), name=f'Forte({n_iter})')
    wins = 0
    for k in range(REPS):
        ai1, ai2 = (strong, baseline) if k%2==0 else (baseline, strong)
        p_strong = PLAYER1 if k%2==0 else PLAYER2
        w, _ = auto_play(ai1, ai2)
        if w == p_strong: wins += 1
    print(f'  n={n_iter:>4}: {wins}/{REPS} vitórias  ({100*wins/REPS:.0f}%)')

Taxa de vitória do MCTS Standard(n) vs Fraco(50) em 8 jogos cada:


KeyboardInterrupt: 

### 5.2 Visualização das estatísticas UCT

Para ganhar intuição sobre como o MCTS distribui o seu orçamento de simulações, podemos examinar as estatísticas de cada jogada possível a partir da posição inicial. Numa posição simétrica como o tabuleiro vazio, esperamos que a coluna central receba mais visitas (por ser estrategicamente mais forte no Connect-4 e variantes).

In [ ]:
# Distribuição de visitas/win-rate para cada jogada possível a partir do estado inicial
start = PopOutGame()
ai = MCTS(iterations=1000, C=math.sqrt(2), name='Stats')
stats = ai.get_move_stats(start)

print('Estatísticas por jogada a partir da posição inicial (X a jogar):')
print(f'{"Jogada":<12} {"Visitas":>8} {"Win-rate":>10}')
print('-' * 32)
for move, s in sorted(stats.items(), key=lambda x: -x[1]['visits']):
    mtype, col = move
    print(f'{mtype.upper()+" col "+str(col+1):<12} {s["visits"]:>8} {s["win_rate"]:>10.3f}')

---
## 6. Geração de Dataset para Árvores de Decisão

A segunda parte do projeto consiste em treinar uma árvore de decisão que consiga prever a jogada recomendada pelo MCTS dado um estado do tabuleiro. Para isso, precisamos primeiro de gerar um dataset de pares `(estado, jogada_ótima)`.

### 6.1 Codificação do Estado

Cada posição do tabuleiro 6x7 é codificada como um vetor de 42 valores inteiros, sempre **da perspetiva do jogador atual**:
- `0` = célula vazia
- `1` = disco do jogador atual
- `2` = disco do adversário

Esta normalização é fundamental: ao treinar a árvore de decisão, o modelo aprende padrões que se aplicam independentemente de qual jogador está a jogar. Sem esta codificação relativa, a árvore teria de aprender regras duplicadas — umas para o Jogador 1, outras para o Jogador 2.

### 6.2 Rótulo da Jogada

O rótulo (label) combina o tipo de jogada com a coluna: `drop_3`, `pop_1`, etc. Este formato permite à árvore de decisão capturar tanto a escolha da coluna como a distinção entre *drop* e *pop* numa única classificação.

In [11]:
def encode_state(game):
    """
    Codifica o tabuleiro da perspetiva do JOGADOR ATUAL.
    disco próprio = 1, adversário = 2, vazio = 0.
    Devolve uma lista plana de 42 inteiros.
    """
    p = game.current_player
    opp = PLAYER2 if p == PLAYER1 else PLAYER1
    enc = []
    for row in game.board:
        for cell in row:
            if cell == p:   enc.append(1)
            elif cell == opp: enc.append(2)
            else:           enc.append(0)
    return enc

def generate_dataset(n_games=50, mcts_iterations=300, output_csv='datasets/popout_dataset.csv'):
    """
    Joga n_games partidas de self-play com MCTS e regista pares (estado, jogada).
    Cada estado é codificado como vetor de 42 dimensões; cada jogada é uma string.
    """
    ai = MCTS(iterations=mcts_iterations, C=math.sqrt(2), rollout='heuristic', name='DataGen')
    rows = []
    feature_names = [f'cell_{r}_{c}' for r in range(6) for c in range(7)]

    for game_idx in range(n_games):
        game = PopOutGame()
        while not game.game_over:
            if game.is_repetition_draw_available():
                break
            move = ai.get_best_move(game)
            if move is None: break
            state_enc = encode_state(game)
            move_label = f'{move[0]}_{move[1]}'
            rows.append(state_enc + [move_label])
            ng = game.apply_move(*move)
            if ng is None: break
            game = ng
        if (game_idx + 1) % 10 == 0:
            print(f'  Gerados {game_idx+1}/{n_games} jogos  ({len(rows)} amostras até agora)')

    # Gravar CSV
    header = feature_names + ['move']
    with open(output_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)

    print(f'\nDataset gravado em {output_csv}  ({len(rows)} amostras)')
    return rows, header

print('A gerar dataset (200 jogos, pode demorar ~2-3 min)...')
t0 = time.time()
dataset_rows, header = generate_dataset(n_games=200, mcts_iterations=400)
print(f'Concluído em {time.time()-t0:.1f}s')

A gerar dataset (200 jogos, pode demorar ~2-3 min)...


KeyboardInterrupt: 

In [ ]:
# Inspeção rápida do dataset
labels = [r[-1] for r in dataset_rows]
label_counts = {}
for l in labels: label_counts[l] = label_counts.get(l, 0) + 1

print(f'Total de amostras : {len(dataset_rows)}')
print(f'Jogadas únicas    : {len(label_counts)}')
print('\nDistribuição de jogadas (top 15):')
for move, cnt in sorted(label_counts.items(), key=lambda x: -x[1])[:15]:
    bar = '#' * (cnt // 5)
    print(f'  {move:<12} {cnt:>5}  {bar}')

---
## 7. Árvores de Decisão — Algoritmo ID3

### 7.1 Enquadramento Teórico

As árvores de decisão são modelos de aprendizagem supervisionada que particionam os dados recursivamente com base em regras simples. O algoritmo **ID3** (*Iterative Dichotomiser 3*), proposto por Quinlan (1986), constrói a árvore selecionando em cada nó o atributo que maximiza o **ganho de informação** — ou seja, o atributo que mais reduz a **entropia** do conjunto de dados.

#### Entropia

A entropia mede a incerteza (ou "desordem") de um conjunto de rótulos:

$$H(S) = -\sum_{c \in C} p_c \cdot \log_2(p_c)$$

onde $p_c$ é a proporção de exemplos da classe $c$. Um conjunto com apenas uma classe tem entropia 0 (certeza total); um conjunto uniformemente distribuído tem entropia máxima.

#### Ganho de Informação

O ganho de informação de um atributo $A$ mede quanta entropia é eliminada ao dividir os dados segundo $A$:

$$IG(S, A) = H(S) - \sum_{v \in \text{valores}(A)} \frac{|S_v|}{|S|} \cdot H(S_v)$$

O ID3 escolhe, em cada passo, o atributo com maior ganho de informação. Este processo repete-se recursivamente até que todas as folhas tenham uma classe pura, não existam mais atributos, ou se atinja uma profundidade máxima.

### 7.2 Tratamento de Atributos Contínuos

O ID3 original trabalha apenas com atributos categóricos. Para lidar com atributos numéricos (como os valores 0, 1, 2 do tabuleiro, ou os atributos contínuos do dataset Iris), implementámos a seguinte estratégia: para cada atributo contínuo, ordenam-se os valores únicos e testam-se todos os pontos médios entre valores consecutivos como possíveis limiares de divisão. O limiar que produz o maior ganho de informação é selecionado.

### 7.3 Restrição Importante

Conforme exigido no enunciado, **não utilizámos scikit-learn nem outras bibliotecas** para definir ou treinar as árvores de decisão. Toda a implementação foi feita de raiz.

In [ ]:
# ─── Implementação do ID3 ────────────────────────────────────────────────────

from collections import Counter

class ID3Tree:
    """
    Árvore de decisão construída pelo algoritmo ID3.
    Suporta atributos categóricos e contínuos (com discretização automática).
    """

    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None
        self.attributes = None

    # ── Entropia ──────────────────────────────────────────────────────────────

    @staticmethod
    def entropy(labels):
        """Calcula a entropia de Shannon para uma lista de rótulos."""
        n = len(labels)
        if n == 0:
            return 0.0
        counts = Counter(labels)
        ent = 0.0
        for count in counts.values():
            p = count / n
            if p > 0:
                ent -= p * math.log2(p)
        return ent

    # ── Ganho de informação (atributos categóricos) ───────────────────────────

    def _info_gain_categorical(self, data, labels, attr_idx):
        """Ganho de informação para um atributo categórico."""
        parent_entropy = self.entropy(labels)
        values = set(row[attr_idx] for row in data)
        weighted_entropy = 0.0
        for val in values:
            subset_labels = [labels[i] for i, row in enumerate(data) if row[attr_idx] == val]
            weighted_entropy += (len(subset_labels) / len(labels)) * self.entropy(subset_labels)
        return parent_entropy - weighted_entropy, None

    # ── Ganho de informação (atributos contínuos) ─────────────────────────────

    def _info_gain_continuous(self, data, labels, attr_idx):
        """Ganho de informação para um atributo contínuo, testando todos os limiares."""
        parent_entropy = self.entropy(labels)
        values = sorted(set(row[attr_idx] for row in data))
        if len(values) <= 1:
            return -1, None

        best_gain = -1
        best_threshold = None
        for k in range(len(values) - 1):
            threshold = (values[k] + values[k + 1]) / 2.0
            left_labels  = [labels[i] for i, row in enumerate(data) if row[attr_idx] <= threshold]
            right_labels = [labels[i] for i, row in enumerate(data) if row[attr_idx] > threshold]
            if not left_labels or not right_labels:
                continue
            w_entropy = (len(left_labels)/len(labels)) * self.entropy(left_labels) + \
                        (len(right_labels)/len(labels)) * self.entropy(right_labels)
            gain = parent_entropy - w_entropy
            if gain > best_gain:
                best_gain = gain
                best_threshold = threshold

        return best_gain, best_threshold

    # ── Construção da árvore ──────────────────────────────────────────────────

    def fit(self, data, labels, attribute_names=None, continuous_attrs=None):
        """
        Treina a árvore de decisão.
        - data: lista de listas (cada sublista é um exemplo)
        - labels: lista de rótulos correspondentes
        - attribute_names: nomes dos atributos (opcional)
        - continuous_attrs: conjunto de índices de atributos contínuos
        """
        n_attrs = len(data[0]) if data else 0
        self.attributes = attribute_names or [f'attr_{i}' for i in range(n_attrs)]
        self.continuous_attrs = continuous_attrs or set()
        available = list(range(n_attrs))
        self.tree = self._build(data, labels, available, depth=0)

    def _build(self, data, labels, available_attrs, depth):
        """Construção recursiva da árvore."""
        # Caso base: todos os exemplos têm o mesmo rótulo
        if len(set(labels)) == 1:
            return labels[0]

        # Caso base: sem atributos disponíveis ou profundidade máxima atingida
        if not available_attrs or (self.max_depth is not None and depth >= self.max_depth):
            return Counter(labels).most_common(1)[0][0]

        # Caso base: poucas amostras para dividir
        if len(data) < self.min_samples_split:
            return Counter(labels).most_common(1)[0][0]

        # Selecionar o melhor atributo
        best_attr = None
        best_gain = -1
        best_threshold = None

        for attr_idx in available_attrs:
            if attr_idx in self.continuous_attrs:
                gain, threshold = self._info_gain_continuous(data, labels, attr_idx)
            else:
                gain, threshold = self._info_gain_categorical(data, labels, attr_idx)
            if gain > best_gain:
                best_gain = gain
                best_attr = attr_idx
                best_threshold = threshold

        if best_attr is None or best_gain <= 0:
            return Counter(labels).most_common(1)[0][0]

        attr_name = self.attributes[best_attr]

        # Divisão contínua (binária)
        if best_attr in self.continuous_attrs and best_threshold is not None:
            left_data   = [row for row in data if row[best_attr] <= best_threshold]
            left_labels = [labels[i] for i, row in enumerate(data) if row[best_attr] <= best_threshold]
            right_data   = [row for row in data if row[best_attr] > best_threshold]
            right_labels = [labels[i] for i, row in enumerate(data) if row[best_attr] > best_threshold]

            node = {
                '_attr': best_attr,
                '_attr_name': attr_name,
                '_threshold': best_threshold,
                '_type': 'continuous'
            }
            if left_data:
                node['<= ' + str(round(best_threshold, 3))] = self._build(
                    left_data, left_labels, available_attrs, depth + 1)
            if right_data:
                node['> ' + str(round(best_threshold, 3))] = self._build(
                    right_data, right_labels, available_attrs, depth + 1)
            return node

        # Divisão categórica
        values = set(row[best_attr] for row in data)
        remaining_attrs = [a for a in available_attrs if a != best_attr]

        node = {
            '_attr': best_attr,
            '_attr_name': attr_name,
            '_type': 'categorical'
        }
        for val in values:
            subset_data   = [row for i, row in enumerate(data) if row[best_attr] == val]
            subset_labels = [labels[i] for i, row in enumerate(data) if row[best_attr] == val]
            if subset_data:
                node[val] = self._build(subset_data, subset_labels, remaining_attrs, depth + 1)
            else:
                node[val] = Counter(labels).most_common(1)[0][0]

        # Guardar classe maioritária como fallback
        node['_default'] = Counter(labels).most_common(1)[0][0]
        return node

    # ── Classificação ─────────────────────────────────────────────────────────

    def predict_one(self, example, node=None):
        """Classifica um único exemplo percorrendo a árvore."""
        if node is None:
            node = self.tree
        if not isinstance(node, dict):
            return node

        attr_idx = node['_attr']
        val = example[attr_idx]

        if node['_type'] == 'continuous':
            threshold = node['_threshold']
            key = '<= ' + str(round(threshold, 3)) if val <= threshold else '> ' + str(round(threshold, 3))
            if key in node:
                return self.predict_one(example, node[key])
            return node.get('_default', None)
        else:
            if val in node:
                return self.predict_one(example, node[val])
            return node.get('_default', None)

    def predict(self, data):
        """Classifica uma lista de exemplos."""
        return [self.predict_one(row) for row in data]

    # ── Métricas ──────────────────────────────────────────────────────────────

    @staticmethod
    def accuracy(y_true, y_pred):
        return sum(1 for a, b in zip(y_true, y_pred) if a == b) / len(y_true)

    @staticmethod
    def precision_recall_f1(y_true, y_pred):
        """Calcula precision, recall e F1-score (macro-average)."""
        classes = set(y_true) | set(y_pred)
        precisions, recalls = [], []
        for cls in classes:
            tp = sum(1 for a, b in zip(y_true, y_pred) if a == cls and b == cls)
            fp = sum(1 for a, b in zip(y_true, y_pred) if a != cls and b == cls)
            fn = sum(1 for a, b in zip(y_true, y_pred) if a == cls and b != cls)
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            precisions.append(prec)
            recalls.append(rec)
        avg_prec = sum(precisions) / len(precisions) if precisions else 0.0
        avg_rec  = sum(recalls) / len(recalls) if recalls else 0.0
        f1 = 2 * avg_prec * avg_rec / (avg_prec + avg_rec) if (avg_prec + avg_rec) > 0 else 0.0
        return avg_prec, avg_rec, f1

    # ── Visualização da árvore ────────────────────────────────────────────────

    def print_tree(self, node=None, indent='', max_depth=5, depth=0):
        """Imprime a árvore de forma legível."""
        if node is None:
            node = self.tree
        if not isinstance(node, dict):
            print(f'{indent}=> {node}')
            return
        if depth >= max_depth:
            print(f'{indent}=> (...)  [cortado a profundidade {max_depth}]')
            return
        attr_name = node.get('_attr_name', '?')
        for key, subtree in node.items():
            if key.startswith('_'):
                continue
            print(f'{indent}[{attr_name} = {key}]')
            self.print_tree(subtree, indent + '  ', max_depth, depth + 1)

    def count_nodes(self, node=None):
        """Conta o número total de nós na árvore."""
        if node is None:
            node = self.tree
        if not isinstance(node, dict):
            return 1
        total = 1
        for key, subtree in node.items():
            if not key.startswith('_'):
                total += self.count_nodes(subtree)
        return total

    def get_depth(self, node=None):
        """Calcula a profundidade máxima da árvore."""
        if node is None:
            node = self.tree
        if not isinstance(node, dict):
            return 0
        max_d = 0
        for key, subtree in node.items():
            if not key.startswith('_'):
                max_d = max(max_d, self.get_depth(subtree))
        return 1 + max_d

print('ID3Tree definido.')

---
## 8. Validação com o Dataset Iris

Antes de aplicar o ID3 ao problema do PopOut, validamos a implementação com o **dataset Iris** — um benchmark clássico de classificação com 150 amostras, 4 atributos contínuos e 3 classes (*Iris setosa*, *Iris versicolor*, *Iris virginica*).

Este teste serve dois propósitos:
1. **Verificação funcional** — confirmar que o ID3 com tratamento de atributos contínuos funciona corretamente
2. **Referência de desempenho** — um dataset bem estudado permite comparar os nossos resultados com valores conhecidos da literatura

In [ ]:
# ─── Visualização Gráfica da Árvore de Decisão ───────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def _count_leaves(node, max_d, depth=0):
    """Conta folhas até profundidade max_d para calcular largura do plot."""
    if not isinstance(node, dict) or (max_d is not None and depth >= max_d):
        return 1
    children = [(k, v) for k, v in node.items() if not (isinstance(k, str) and k.startswith('_'))]
    if not children:
        return 1
    return sum(_count_leaves(v, max_d, depth + 1) for _, v in children)

def _draw_tree(ax, node, x, y, dx, dy, max_d, depth=0, parent_xy=None):
    """Desenha recursivamente nós e ligações."""
    if parent_xy:
        ax.plot([parent_xy[0], x], [parent_xy[1] - 0.05, y + 0.05],
                color='#888888', lw=0.8, zorder=1)

    if not isinstance(node, dict) or (max_d is not None and depth >= max_d):
        # Folha
        label = str(node) if not isinstance(node, dict) else node.get('_default', '?')
        ax.text(x, y, label, ha='center', va='center', fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#a8e6a3', edgecolor='#2e7d32', lw=1),
                zorder=2)
        return

    attr_name = node.get('_attr_name', '?')
    thr       = node.get('_threshold')
    if thr is not None:
        label = f"{attr_name}\n≤ {round(thr, 2)}"
    else:
        label = attr_name

    ax.text(x, y, label, ha='center', va='center', fontsize=7,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#a3c8e6', edgecolor='#1565c0', lw=1),
            zorder=2)

    children = sorted([(k, v) for k, v in node.items()
                        if not (isinstance(k, str) and k.startswith('_'))],
                       key=lambda x: str(x[0]))
    if not children:
        return

    # Distribuir filhos horizontalmente
    n_leaves = [_count_leaves(v, max_d, depth + 1) for _, v in children]
    total    = sum(n_leaves)
    x_start  = x - dx / 2
    for (k, child), nl in zip(children, n_leaves):
        child_x = x_start + (nl / total) * dx / 2
        # Rótulo da aresta
        edge_label = str(k) if not isinstance(k, str) or not k.startswith('_') else ''
        mx = (x + child_x) / 2
        my = (y + (y - dy)) / 2
        if edge_label:
            ax.text(mx, my, edge_label, ha='center', va='center', fontsize=5.5,
                    color='#555555', zorder=3)
        _draw_tree(ax, child, child_x, y - dy, nl / total * dx, dy, max_d, depth + 1, (x, y))
        x_start += (nl / total) * dx

def plot_tree(tree_obj, title='Árvore de Decisão', max_depth=4, figsize=(14, 6)):
    """
    Visualização gráfica da árvore de decisão.
    tree_obj : instância de ID3Tree
    max_depth: profundidade máxima a desenhar
    """
    root = tree_obj.tree
    n_leaves = _count_leaves(root, max_depth)
    fig_w = max(figsize[0], n_leaves * 1.2)
    fig, ax = plt.subplots(figsize=(fig_w, figsize[1]))
    ax.set_xlim(0, 1)
    ax.set_ylim(-max_depth - 0.5, 0.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

    _draw_tree(ax, root, x=0.5, y=0, dx=1.0, dy=0.9, max_d=max_depth)

    legend = [
        mpatches.Patch(facecolor='#a3c8e6', edgecolor='#1565c0', label='Nó interno (divisão)'),
        mpatches.Patch(facecolor='#a8e6a3', edgecolor='#2e7d32', label='Folha (classe)'),
    ]
    ax.legend(handles=legend, loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()

print("Função plot_tree definida.")


In [ ]:
# ─── Carregamento e preparação do dataset Iris ────────────────────────────────

def load_iris_csv(filepath='datasets/iris.csv'):
    """
    Carrega o dataset Iris a partir de CSV.
    Tenta ler o ficheiro; se não existir, usa os dados embutidos como fallback.
    """
    import csv, os
    if os.path.exists(filepath):
        with open(filepath, newline='', encoding='utf-8') as f_csv:
            reader = csv.reader(f_csv)
            header = next(reader)
            rows   = list(reader)
        # formato: id, sepal_length, sepal_width, petal_length, petal_width, class
        data   = [[float(r[1]), float(r[2]), float(r[3]), float(r[4])] for r in rows]
        labels = [r[5].strip() for r in rows]
        return data, labels
    # ── Fallback: dados embutidos inline ────────────────────────────────────
    iris_data_raw = [
        # sepal_length, sepal_width, petal_length, petal_width, class
        [5.1,3.5,1.4,0.2,'setosa'],[4.9,3.0,1.4,0.2,'setosa'],[4.7,3.2,1.3,0.2,'setosa'],
        [4.6,3.1,1.5,0.2,'setosa'],[5.0,3.6,1.4,0.2,'setosa'],[5.4,3.9,1.7,0.4,'setosa'],
        [4.6,3.4,1.4,0.3,'setosa'],[5.0,3.4,1.5,0.2,'setosa'],[4.4,2.9,1.4,0.2,'setosa'],
        [4.9,3.1,1.5,0.1,'setosa'],[5.4,3.7,1.5,0.2,'setosa'],[4.8,3.4,1.6,0.2,'setosa'],
        [4.8,3.0,1.4,0.1,'setosa'],[4.3,3.0,1.1,0.1,'setosa'],[5.8,4.0,1.2,0.2,'setosa'],
        [5.7,4.4,1.5,0.4,'setosa'],[5.4,3.9,1.3,0.4,'setosa'],[5.1,3.5,1.4,0.3,'setosa'],
        [5.7,3.8,1.7,0.3,'setosa'],[5.1,3.8,1.5,0.3,'setosa'],[5.4,3.4,1.7,0.2,'setosa'],
        [5.1,3.7,1.5,0.4,'setosa'],[4.6,3.6,1.0,0.2,'setosa'],[5.1,3.3,1.7,0.5,'setosa'],
        [4.8,3.4,1.9,0.2,'setosa'],[5.0,3.0,1.6,0.2,'setosa'],[5.0,3.4,1.6,0.4,'setosa'],
        [5.2,3.5,1.5,0.2,'setosa'],[5.2,3.4,1.4,0.2,'setosa'],[4.7,3.2,1.6,0.2,'setosa'],
        [4.8,3.1,1.6,0.2,'setosa'],[5.4,3.4,1.5,0.4,'setosa'],[5.2,4.1,1.5,0.1,'setosa'],
        [5.5,4.2,1.4,0.2,'setosa'],[4.9,3.1,1.5,0.2,'setosa'],[5.0,3.2,1.2,0.2,'setosa'],
        [5.5,3.5,1.3,0.2,'setosa'],[4.9,3.6,1.4,0.1,'setosa'],[4.4,3.0,1.3,0.2,'setosa'],
        [5.1,3.4,1.5,0.2,'setosa'],[5.0,3.5,1.3,0.3,'setosa'],[4.5,2.3,1.3,0.3,'setosa'],
        [4.4,3.2,1.3,0.2,'setosa'],[5.0,3.5,1.6,0.6,'setosa'],[5.1,3.8,1.9,0.4,'setosa'],
        [4.8,3.0,1.4,0.3,'setosa'],[5.1,3.8,1.6,0.2,'setosa'],[4.6,3.2,1.4,0.2,'setosa'],
        [5.3,3.7,1.5,0.2,'setosa'],[5.0,3.3,1.4,0.2,'setosa'],
        [7.0,3.2,4.7,1.4,'versicolor'],[6.4,3.2,4.5,1.5,'versicolor'],[6.9,3.1,4.9,1.5,'versicolor'],
        [5.5,2.3,4.0,1.3,'versicolor'],[6.5,2.8,4.6,1.5,'versicolor'],[5.7,2.8,4.5,1.3,'versicolor'],
        [6.3,3.3,4.7,1.6,'versicolor'],[4.9,2.4,3.3,1.0,'versicolor'],[6.6,2.9,4.6,1.3,'versicolor'],
        [5.2,2.7,3.9,1.4,'versicolor'],[5.0,2.0,3.5,1.0,'versicolor'],[5.9,3.0,4.2,1.5,'versicolor'],
        [6.0,2.2,4.0,1.0,'versicolor'],[6.1,2.9,4.7,1.4,'versicolor'],[5.6,2.9,3.6,1.3,'versicolor'],
        [6.7,3.1,4.4,1.4,'versicolor'],[5.6,3.0,4.5,1.5,'versicolor'],[5.8,2.7,4.1,1.0,'versicolor'],
        [6.2,2.2,4.5,1.5,'versicolor'],[5.6,2.5,3.9,1.1,'versicolor'],[5.9,3.2,4.8,1.8,'versicolor'],
        [6.1,2.8,4.0,1.3,'versicolor'],[6.3,2.5,4.9,1.5,'versicolor'],[6.1,2.8,4.7,1.2,'versicolor'],
        [6.4,2.9,4.3,1.3,'versicolor'],[6.6,3.0,4.4,1.4,'versicolor'],[6.8,2.8,4.8,1.4,'versicolor'],
        [6.7,3.0,5.0,1.7,'versicolor'],[6.0,2.9,4.5,1.5,'versicolor'],[5.7,2.6,3.5,1.0,'versicolor'],
        [5.5,2.4,3.8,1.1,'versicolor'],[5.5,2.4,3.7,1.0,'versicolor'],[5.8,2.7,3.9,1.2,'versicolor'],
        [6.0,2.7,5.1,1.6,'versicolor'],[5.4,3.0,4.5,1.5,'versicolor'],[6.0,3.4,4.5,1.6,'versicolor'],
        [6.7,3.1,4.7,1.5,'versicolor'],[6.3,2.3,4.4,1.3,'versicolor'],[5.6,3.0,4.1,1.3,'versicolor'],
        [5.5,2.5,4.0,1.3,'versicolor'],[5.5,2.6,4.4,1.2,'versicolor'],[6.1,3.0,4.6,1.4,'versicolor'],
        [5.8,2.6,4.0,1.2,'versicolor'],[5.0,2.3,3.3,1.0,'versicolor'],[5.6,2.7,4.2,1.3,'versicolor'],
        [5.7,3.0,4.2,1.2,'versicolor'],[5.7,2.9,4.2,1.3,'versicolor'],[6.2,2.9,4.3,1.3,'versicolor'],
        [5.1,2.5,3.0,1.1,'versicolor'],[5.7,2.8,4.1,1.3,'versicolor'],
        [6.3,3.3,6.0,2.5,'virginica'],[5.8,2.7,5.1,1.9,'virginica'],[7.1,3.0,5.9,2.1,'virginica'],
        [6.3,2.9,5.6,1.8,'virginica'],[6.5,3.0,5.8,2.2,'virginica'],[7.6,3.0,6.6,2.1,'virginica'],
        [4.9,2.5,4.5,1.7,'virginica'],[7.3,2.9,6.3,1.8,'virginica'],[6.7,2.5,5.8,1.8,'virginica'],
        [7.2,3.6,6.1,2.5,'virginica'],[6.5,3.2,5.1,2.0,'virginica'],[6.4,2.7,5.3,1.9,'virginica'],
        [6.8,3.0,5.5,2.1,'virginica'],[5.7,2.5,5.0,2.0,'virginica'],[5.8,2.8,5.1,2.4,'virginica'],
        [6.4,3.2,5.3,2.3,'virginica'],[6.5,3.0,5.5,1.8,'virginica'],[7.7,3.8,6.7,2.2,'virginica'],
        [7.7,2.6,6.9,2.3,'virginica'],[6.0,2.2,5.0,1.5,'virginica'],[6.9,3.2,5.7,2.3,'virginica'],
        [5.6,2.8,4.9,2.0,'virginica'],[7.7,2.8,6.7,2.0,'virginica'],[6.3,2.7,4.9,1.8,'virginica'],
        [6.7,3.3,5.7,2.1,'virginica'],[7.2,3.2,6.0,1.8,'virginica'],[6.2,2.8,4.8,1.8,'virginica'],
        [6.1,3.0,4.9,1.8,'virginica'],[6.4,2.8,5.6,2.1,'virginica'],[7.2,3.0,5.8,1.6,'virginica'],
        [7.4,2.8,6.1,1.9,'virginica'],[7.9,3.8,6.4,2.0,'virginica'],[6.4,2.8,5.6,2.2,'virginica'],
        [6.3,2.8,5.1,1.5,'virginica'],[6.1,2.6,5.6,1.4,'virginica'],[7.7,3.0,6.1,2.3,'virginica'],
        [6.3,3.4,5.6,2.4,'virginica'],[6.4,3.1,5.5,1.8,'virginica'],[6.0,3.0,4.8,1.8,'virginica'],
        [6.9,3.1,5.4,2.1,'virginica'],[6.7,3.1,5.6,2.4,'virginica'],[6.9,3.1,5.1,2.3,'virginica'],
        [5.8,2.7,5.1,1.9,'virginica'],[6.8,3.2,5.9,2.3,'virginica'],[6.7,3.3,5.7,2.5,'virginica'],
        [6.7,3.0,5.2,2.3,'virginica'],[6.3,2.5,5.0,1.9,'virginica'],[6.5,3.0,5.2,2.0,'virginica'],
        [6.2,3.4,5.4,2.3,'virginica'],[5.9,3.0,5.1,1.8,'virginica']
    ]
    data   = [row[:4] for row in iris_data_raw]
    labels = [row[4] for row in iris_data_raw]
    return data, labels

iris_data, iris_labels = load_iris_csv()
print(f'Iris dataset: {len(iris_data)} amostras, {len(iris_data[0])} atributos')
print(f'Classes: {set(iris_labels)}')

In [ ]:
# ─── Treino e teste no Iris (divisão 80/20) ──────────────────────────────────

def train_test_split(data, labels, test_ratio=0.2, seed=42):
    """Divisão simples em treino/teste com shuffle determinístico."""
    indices = list(range(len(data)))
    rng = random.Random(seed)
    rng.shuffle(indices)
    split = int(len(data) * (1 - test_ratio))
    train_idx, test_idx = indices[:split], indices[split:]
    return ([data[i] for i in train_idx], [labels[i] for i in train_idx],
            [data[i] for i in test_idx],  [labels[i] for i in test_idx])

iris_attr_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
# Todos os atributos do Iris são contínuos
iris_continuous = {0, 1, 2, 3}

X_train, y_train, X_test, y_test = train_test_split(iris_data, iris_labels)

iris_tree = ID3Tree(max_depth=10)
iris_tree.fit(X_train, y_train, attribute_names=iris_attr_names, continuous_attrs=iris_continuous)

# Avaliação
train_pred = iris_tree.predict(X_train)
test_pred  = iris_tree.predict(X_test)

train_acc = ID3Tree.accuracy(y_train, train_pred)
test_acc  = ID3Tree.accuracy(y_test, test_pred)
prec, rec, f1 = ID3Tree.precision_recall_f1(y_test, test_pred)

print('=== Resultados no Dataset Iris ===')
print(f'Accuracy (treino): {train_acc:.4f}')
print(f'Accuracy (teste):  {test_acc:.4f}')
print(f'Precision (macro): {prec:.4f}')
print(f'Recall (macro):    {rec:.4f}')
print(f'F1-score (macro):  {f1:.4f}')
print(f'\nNós na árvore: {iris_tree.count_nodes()}')
print(f'Profundidade:  {iris_tree.get_depth()}')

In [ ]:
# Visualização parcial da árvore Iris
print('Árvore de decisão Iris (primeiros 4 níveis):')
print()
iris_tree.print_tree(max_depth=4)

### 8.1 Análise dos Resultados — Iris

Os resultados no Iris servem como validação da nossa implementação. A classe *Iris setosa* é linearmente separável das outras duas (basta o comprimento da pétala), o que explica porque a árvore a consegue classificar com facilidade. A dificuldade está na fronteira entre *versicolor* e *virginica*, que se sobrepõem parcialmente.

Se a accuracy de treino for significativamente superior à de teste, isto indica **overfitting** — a árvore memorizou particularidades do conjunto de treino em vez de aprender padrões gerais. Este fenómeno é esperado no ID3 puro (sem poda), e será igualmente relevante quando aplicarmos o modelo ao PopOut.

In [ ]:
# Visualização gráfica da árvore Iris (até 4 níveis)
plot_tree(iris_tree, title='Árvore de Decisão — Dataset Iris (max 4 níveis)', max_depth=4)


---
## 9. Árvore de Decisão para o PopOut

Aplicamos agora o ID3 ao dataset gerado na secção 6. O objetivo é treinar uma árvore que, dado um estado do tabuleiro (42 features), preveja a jogada que o MCTS recomendaria.

### 9.1 Desafios Específicos

Este é um problema consideravelmente mais difícil do que o Iris:
- **42 atributos** (vs. 4 no Iris), todos discretos com valores {0, 1, 2}
- **Número de classes elevado** — até 14 jogadas distintas (7 drops + 7 pops), embora nem todas ocorram com frequência
- **Relações espaciais complexas** — a "melhor jogada" depende de padrões globais no tabuleiro que são difíceis de capturar com divisões binárias simples
- **Distribuição desbalanceada** — *drops* nas colunas centrais dominam o dataset, enquanto *pops* são raros no início do jogo

Estas limitações são inerentes à natureza do ID3, que opera com divisões univariadas e não captura interações entre features sem profundidade excessiva.

In [ ]:
# ─── Preparar dados do PopOut para o ID3 ─────────────────────────────────────

popout_data   = [row[:-1] for row in dataset_rows]  # 42 features
popout_labels = [row[-1] for row in dataset_rows]     # rótulo: 'drop_3', 'pop_1', etc.

feature_names = [f'cell_{r}_{c}' for r in range(6) for c in range(7)]

# Divisão treino/teste
X_tr, y_tr, X_te, y_te = train_test_split(popout_data, popout_labels, test_ratio=0.2, seed=42)

print(f'Treino: {len(X_tr)} amostras')
print(f'Teste:  {len(X_te)} amostras')
print(f'Classes únicas: {len(set(popout_labels))}')

In [ ]:
# ─── Treinar ID3 no dataset PopOut ───────────────────────────────────────────

# Atributos categóricos (valores discretos 0, 1, 2)
popout_tree = ID3Tree(max_depth=20, min_samples_split=5)
t0 = time.time()
popout_tree.fit(X_tr, y_tr, attribute_names=feature_names, continuous_attrs=set())
train_time = time.time() - t0

# Avaliação
train_pred = popout_tree.predict(X_tr)
test_pred  = popout_tree.predict(X_te)

train_acc = ID3Tree.accuracy(y_tr, train_pred)
test_acc  = ID3Tree.accuracy(y_te, test_pred)
prec, rec, f1 = ID3Tree.precision_recall_f1(y_te, test_pred)

print('=== Resultados no Dataset PopOut ===')
print(f'Tempo de treino:   {train_time:.2f}s')
print(f'Accuracy (treino): {train_acc:.4f}')
print(f'Accuracy (teste):  {test_acc:.4f}')
print(f'Precision (macro): {prec:.4f}')
print(f'Recall (macro):    {rec:.4f}')
print(f'F1-score (macro):  {f1:.4f}')
print(f'\nNós na árvore:     {popout_tree.count_nodes()}')
print(f'Profundidade:      {popout_tree.get_depth()}')

In [ ]:
# Visualização parcial da árvore PopOut
print('Árvore de decisão PopOut (primeiros 3 níveis):')
print()
popout_tree.print_tree(max_depth=3)

### 9.2 Efeito da profundidade máxima

Para compreender melhor a relação entre complexidade do modelo e capacidade de generalização, treinamos árvores com diferentes profundidades máximas e comparamos accuracy de treino e teste.

In [ ]:
# Testar diferentes profundidades
depths = [3, 5, 8, 10, 15, 20, None]  # None = sem limite

print(f'{"Profundidade":<14} {"Acc Treino":>12} {"Acc Teste":>12} {"Nós":>8}')
print('-' * 50)

for d in depths:
    tree = ID3Tree(max_depth=d, min_samples_split=3)
    tree.fit(X_tr, y_tr, attribute_names=feature_names, continuous_attrs=set())
    tr_acc = ID3Tree.accuracy(y_tr, tree.predict(X_tr))
    te_acc = ID3Tree.accuracy(y_te, tree.predict(X_te))
    n_nodes = tree.count_nodes()
    d_str = str(d) if d is not None else 'Sem limite'
    print(f'{d_str:<14} {tr_acc:>12.4f} {te_acc:>12.4f} {n_nodes:>8}')

In [ ]:
# Visualização gráfica da árvore PopOut (até 3 níveis — mais níveis ficam ilegíveis)
plot_tree(popout_tree, title='Árvore de Decisão — Dataset PopOut (max 3 níveis)', max_depth=3)


### 9.2b K-Fold Cross-Validation

Para uma avaliação mais robusta do modelo PopOut, usamos **k-fold cross-validation** (k=5).
A divisão 80/20 simples tem variância elevada quando o dataset é pequeno; o k-fold
usa todas as amostras para treino e teste, de forma rotativa.


In [ ]:
# ─── K-Fold Cross-Validation ─────────────────────────────────────────────────

def kfold_cv(data, labels, k=5, max_depth=10, min_samples=3, seed=42):
    """
    K-fold cross-validation para o ID3Tree.
    Retorna (média da accuracy, lista de accuracies por fold).
    """
    n = len(data)
    indices = list(range(n))
    rng = random.Random(seed)
    rng.shuffle(indices)

    fold_size = n // k
    accs = []

    for fold in range(k):
        val_start = fold * fold_size
        val_end   = val_start + fold_size if fold < k - 1 else n
        val_idx   = set(indices[val_start:val_end])
        train_idx = [i for i in indices if i not in val_idx]
        val_idx   = list(indices[val_start:val_end])

        X_tr = [data[i] for i in train_idx]
        y_tr = [labels[i] for i in train_idx]
        X_vl = [data[i] for i in val_idx]
        y_vl = [labels[i] for i in val_idx]

        tree = ID3Tree(max_depth=max_depth, min_samples_split=min_samples)
        tree.fit(X_tr, y_tr, attribute_names=feature_names, continuous_attrs=set())
        acc = ID3Tree.accuracy(y_vl, tree.predict(X_vl))
        accs.append(acc)

    return sum(accs) / k, accs

print("A correr 5-fold CV (max_depth=10)…")
mean_acc, fold_accs = kfold_cv(popout_data, popout_labels, k=5, max_depth=10)

print(f"\n5-Fold Cross-Validation — PopOut")
print(f"{'Fold':<8} {'Accuracy':>10}")
print("-" * 20)
for i, acc in enumerate(fold_accs, 1):
    print(f"Fold {i:<4} {acc:>10.4f}")
print("-" * 20)
print(f"{'Média':<8} {mean_acc:>10.4f}")
print(f"{'Std':<8} {(sum((a-mean_acc)**2 for a in fold_accs)/len(fold_accs))**0.5:>10.4f}")


### 9.3 Análise dos Resultados — PopOut

Como esperado, o desempenho no dataset PopOut é inferior ao do Iris. Vários fatores contribuem:

1. **O problema é inerentemente difícil para árvores de decisão** — a "melhor jogada" depende de padrões complexos que envolvem múltiplas células em simultâneo. O ID3, ao dividir por um atributo de cada vez, precisa de grande profundidade para capturar estas interações.

2. **Overfitting** — é provável que a accuracy de treino seja muito superior à de teste, indicando que a árvore memoriza o dataset em vez de generalizar. Técnicas como poda (*pruning*) ou *ensemble methods* (e.g., bagging) poderiam mitigar este problema.

3. **Tamanho do dataset** — 50 jogos geram tipicamente entre 1000 e 2000 amostras. Para um problema com 42 features e mais de 10 classes, este volume pode ser insuficiente. Aumentar o número de jogos (e.g., para 200-500) deverá melhorar os resultados.

4. **Qualidade dos rótulos** — o MCTS com 300 iterações não é infalível: com este orçamento, a jogada "recomendada" pode variar entre execuções, introduzindo ruído nos dados de treino.

Apesar destas limitações, a árvore consegue acertar acima do que seria esperado por classificação aleatória (que seria ~7-10% para 10-14 classes), demonstrando que captura padrões relevantes do jogo.

### 9.4 Importância das Features — Heatmap do Tabuleiro

Qual é a célula do tabuleiro mais importante para a árvore? Contamos quantas vezes
cada feature `cell_r_c` aparece como nó de divisão na árvore PopOut treinada.
O resultado é visualizado como heatmap 6×7 — revela se a árvore aprendeu que
o centro do tabuleiro é mais informativo (padrão conhecido no Connect-4).


In [ ]:
# ─── Feature Importance via contagem de nós ──────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

def count_feature_splits(node, counts=None):
    """Conta quantas vezes cada feature (índice) é usada como nó de divisão."""
    if counts is None:
        counts = {}
    if not isinstance(node, dict):
        return counts
    fi = node.get('_attr')
    if fi is not None:
        counts[fi] = counts.get(fi, 0) + 1
    for k, child in node.items():
        if not (isinstance(k, str) and k.startswith('_')):
            count_feature_splits(child, counts)
    return counts

feat_counts = count_feature_splits(popout_tree.tree)

# Organizar em grelha 6×7
importance = np.zeros((6, 7))
for fi, cnt in feat_counts.items():
    r, c = divmod(fi, 7)
    importance[r, c] = cnt

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(importance, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Número de nós de divisão')

for r in range(6):
    for c in range(7):
        val = int(importance[r, c])
        ax.text(c, r, str(val) if val > 0 else '', ha='center', va='center',
                fontsize=9, color='black' if importance[r, c] < importance.max()*0.7 else 'white')

ax.set_xticks(range(7)); ax.set_xticklabels([f'Col {i}' for i in range(7)])
ax.set_yticks(range(6)); ax.set_yticklabels([f'Linha {i}' for i in range(6)])
ax.set_title("Importância das Features — Heatmap do Tabuleiro PopOut", fontweight='bold', pad=12)
ax.invert_yaxis()  # linha 0 = topo (como no tabuleiro real)
plt.tight_layout()
plt.show()

top5 = sorted(feat_counts.items(), key=lambda x: -x[1])[:5]
print("Top 5 features mais usadas:")
for fi, cnt in top5:
    r, c = divmod(fi, 7)
    print(f"  cell_{r}_{c}  →  {cnt} divisões")


---
## 10. Dificuldades e Decisões de Projeto

### 10.1 O MCTS "melhora" ao longo do jogo?

Observando jogos IA vs IA, é tentador concluir que o agente MCTS "fica mais inteligente" à medida que o jogo avança. Na realidade, o número de iterações é fixo em cada turno — o que muda é a **dimensão do espaço de pesquisa**. No início do jogo, com 7 colunas vazias, há um vasto número de possibilidades; perto do final, muitas colunas estão cheias e as simulações cobrem uma fração maior do espaço. O resultado é que as decisões finais são mais precisas não porque o agente aprendeu algo, mas porque o problema ficou mais simples.

### 10.2 Regra de empate por tabuleiro cheio

A regra de empate quando o tabuleiro está cheio introduz uma subtileza: se o jogador atual pode fazer *pop*, deve fazê-lo ou declarar empate? Na nossa implementação simplificada, a IA opta sempre por *pop* quando disponível, evitando empates prematuros. Esta abordagem é razoável na maioria dos casos, mas pode levar a situações subótimas quando o *pop* piora a posição.

### 10.3 Custo computacional do MCTS

O principal bottleneck é a fase de simulação, que requer executar jogos completos até ao final. Para 1000 iterações, cada decisão pode exigir milhares de jogadas aleatórias. Mantivemos um cap de 200 movimentos por simulação para evitar ciclos infinitos em posições degeneradas, mas isto significa que simulações muito longas terminam com empate artificial.

### 10.4 Codificação relativa do tabuleiro

A decisão de codificar o tabuleiro da perspetiva do jogador atual (em vez de usar valores absolutos de Jogador 1/Jogador 2) revelou-se crucial para o desempenho da árvore de decisão. Sem esta normalização, a árvore teria de aprender regras separadas para cada jogador, duplicando efetivamente a complexidade do problema.

### 10.5 ID3 sem poda

O ID3 puro não inclui mecanismos de poda, o que o torna vulnerável a overfitting — especialmente com datasets pequenos e ruidosos como o nosso. Implementámos dois mecanismos básicos de controlo: profundidade máxima (`max_depth`) e número mínimo de amostras para dividir (`min_samples_split`). Uma extensão natural seria implementar poda pós-construção ou utilizar o algoritmo C4.5.

---
## 11. Conclusões

### 11.1 MCTS

O algoritmo Monte Carlo Tree Search provou ser eficaz para o PopOut. Com apenas 400 iterações, o agente já demonstra jogo competente contra humanos. As experiências com variantes permitiram observar que:

- A **heuristic rollout** (look-ahead de 1 passo) melhora consistentemente a qualidade das simulações ao evitar erros grosseiros. Quando o agente testa se pode ganhar ou bloquear uma vitória imediata do adversário antes de jogar aleatoriamente, a informação que chega à retropropagação é mais fidedigna.

- O **progressive widening** concentra o orçamento computacional nos ramos mais promissores, o que é especialmente útil nas posições iniciais onde o branching factor é mais elevado.

- A **alta exploração** ($C = 2.5$) tende a dispersar demasiado os recursos com orçamentos limitados. Para tirar partido de valores altos de $C$, seriam necessárias significativamente mais iterações.

- O **UCT-Tuned** mostra o comportamento mais estável, adaptando automaticamente o grau de exploração com base na variância observada.

### 11.2 Árvores de Decisão

A aplicação do ID3 ao PopOut revelou as limitações de modelos univariados em problemas com interações complexas entre features. A accuracy no dataset Iris confirma que a implementação está correta; a queda de desempenho no PopOut reflete a dificuldade intrínseca do problema, não falhas no algoritmo.

Os resultados sugerem que, para melhorar significativamente a capacidade preditiva, seria necessário:
- Aumentar substancialmente o volume de dados de treino
- Introduzir features derivadas (e.g., contagem de ameaças, controlo do centro)
- Explorar técnicas de ensemble como bagging ou random forests
- Implementar poda na árvore para melhorar a generalização

### 11.3 Trabalho Futuro

Ficam como possíveis extensões:
- Implementação de MCTS com *parallel processing* para orçamentos de iteração mais elevados
- Poda da árvore de decisão (C4.5) ou utilização de Rulesets
- Avaliação da árvore de decisão como agente de jogo, medindo o seu desempenho direto contra o MCTS

In [ ]:
# Demonstração final: jogo IA vs IA com as duas variantes de melhor desempenho
ai_best   = MCTS(iterations=600, C=math.sqrt(2), rollout='heuristic', name='Heuristic-Rollout')
ai_tuned  = MCTS(iterations=600, C=math.sqrt(2), tuned=True,         name='UCT-Tuned')

winner, moves = auto_play(ai_best, ai_tuned, verbose=True)
sym = PopOutGame.player_symbol(winner) if winner else 'Empate'
print(f'Resultado final: {sym} em {moves} jogadas')